In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os


Load Pre Filter and Post filter data

In [8]:
# 03 contains all 263 stocks before the beta and thin trading filters
df_all = pd.read_csv('../../output/03_aspi_handled.csv')
df_all['date'] = pd.to_datetime(df_all['date'])

# 04 contains the 237 qualifying stocks
df_clean = pd.read_csv('../../output/04_Regression_handled.csv')

Isolate the excluded stocks

In [9]:
all_symbols = set(df_all['symbol'].unique())
clean_symbols = set(df_clean['symbol'].unique())
excluded_symbols = list(all_symbols - clean_symbols)

print(f"Total symbols initially: {len(all_symbols)}")
print(f"Total symbols after filters: {len(clean_symbols)}")
print(f"Total excluded symbols: {len(excluded_symbols)}")

# Filter the raw data for only the excluded symbols
df_excluded = df_all[df_all['symbol'].isin(excluded_symbols)].copy()

Total symbols initially: 263
Total symbols after filters: 263
Total excluded symbols: 0


Separate 'Thinly Traded' from 'Extreme Beta'

In [10]:
# We count trading days before this date to find the thinly traded ones (< 60 days)
estimation_data = df_excluded[df_excluded['date'] < '2025-11-20']
obs_counts = estimation_data.groupby('symbol').size().reset_index(name='n_obs')

# Classify the exclusions
obs_counts['Reason'] = obs_counts['n_obs'].apply(lambda x: 'Thin Trading (< 60 days)' if x < 60 else 'Extreme Beta Outlier')
extreme_beta_symbols = obs_counts[obs_counts['Reason'] == 'Extreme Beta Outlier']['symbol'].tolist()

# Save the exclusion log for the paper's appendix
obs_counts.to_csv('../../output/tables/08_exclusion_log.csv', index=False)

Generate Diagnostic Plots for Extreme Betas

In [12]:
sns.set_theme(style="whitegrid")

# Define the landfall date to mark on the plots
landfall_date = pd.to_datetime('2025-11-28')

for symbol in extreme_beta_symbols:
    stock_data = df_excluded[df_excluded['symbol'] == symbol].sort_values('date')
    sector = stock_data['sector'].iloc[0]
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), gridspec_kw={'height_ratios': [3, 1]})
    
    # Plot 1: Closing Price
    sns.lineplot(data=stock_data, x='date', y='close', ax=ax1, color='navy')
    ax1.axvline(x=landfall_date, color='red', linestyle='--', label='Cyclone Landfall')
    ax1.set_title(f'Diagnostic Plot: {symbol} ({sector})\nReason: Extreme Beta', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Closing Price (LKR)')
    ax1.legend()

    # Plot 2: Trading Volume
    sns.barplot(data=stock_data, x='date', y='volume', ax=ax2, color='gray', alpha=0.6)
    # Hide x-labels on the volume chart to prevent overlapping text mess
    ax2.set(xticks=[]) 
    ax2.set_ylabel('Volume')
    ax2.set_xlabel('Date (Feb 2025 - Feb 2026)')
    
    plt.tight_layout()
    # Save each plot individually
    file_path = f'../../output/figs/excluded_stocks/{symbol}_diagnostic.png'
    plt.savefig(file_path, dpi=200)
    plt.close()